# Notebook 04d — RL Untrained N=2 Sanity Check

**Gate before training.** Build a 2-asset env (SPY + IEF), run an untrained RLAgent with
stochastic (Dirichlet) sampling across the **test window** (2023–2026), and confirm:
1. SPY weight varies between ~0.1 and ~0.9 across days.
2. Turnover is non-zero and roughly stable.
3. Returns are finite, no NaNs.

If any check fails, halt before proceeding to 04e training.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from aiam.data.split import TEST_START
from aiam.rl.env import PortfolioEnv
from aiam.rl.policy import SimplexPolicy
from aiam.rl.agent import RLAgent
from aiam.rl.trainer import run_episode

LOOKBACK = 20
ASSETS = ['SPY.US', 'IEF.US']

print('Imports OK')

In [ ]:
# Load test-window returns
returns_full = pd.read_parquet('../data/cache/returns_29_2003_2026.parquet')[ASSETS]
returns_test = returns_full.loc[returns_full.index >= TEST_START].dropna()
print(f'Test window: {returns_test.index[0].date()} → {returns_test.index[-1].date()}, {len(returns_test)} days')
print(f'Assets: {ASSETS}')

In [ ]:
# Build env + untrained agent
env = PortfolioEnv(returns_test, lookback=LOOKBACK)
torch.manual_seed(42)
policy = SimplexPolicy(n_features=env.n_features, hidden_dim=32, use_weights=True)
agent = RLAgent(policy, lookback=LOOKBACK, seed=42)
print(f'Env: {env.n_assets} assets, {env.n_features} features/asset')

In [ ]:
# Roll forward with stochastic Dirichlet sampling (temperature=1.0)
np.random.seed(42)
rewards, log_probs, hiddens, turnovers, weights_list = run_episode(
    policy, None, env, temperature=1.0
)

weights_arr = np.array(weights_list)   # (T, 2)
dates = env.dates[env._start_idx + 1 : env._start_idx + 1 + len(weights_list)]

spy_weights = weights_arr[:, 0]
ief_weights = weights_arr[:, 1]

print(f'Steps: {len(rewards)}')
print(f'Returns: finite={np.all(np.isfinite(rewards))}, NaN count={np.sum(np.isnan(rewards))}')
print(f'SPY weight: min={spy_weights.min():.3f}, max={spy_weights.max():.3f}, mean={spy_weights.mean():.3f}')
print(f'Turnover: min={min(turnovers):.4f}, max={max(turnovers):.4f}, mean={np.mean(turnovers):.4f}')

In [ ]:
# GATE CHECKS
assert np.all(np.isfinite(rewards)), 'FAIL: non-finite returns'
assert np.sum(np.isnan(rewards)) == 0, 'FAIL: NaN returns'
assert spy_weights.min() < 0.5, f'FAIL: SPY weight never drops below 0.5 (min={spy_weights.min():.3f})'
assert spy_weights.max() > 0.5, f'FAIL: SPY weight never rises above 0.5 (max={spy_weights.max():.3f})'
assert np.mean(turnovers) > 1e-4, f'FAIL: turnover near zero ({np.mean(turnovers):.6f})'
assert np.std(turnovers) < 1.0, 'FAIL: turnover unstable (std > 1.0)'

print('ALL GATE CHECKS PASSED — proceed to 04e training')

In [ ]:
# Visualize
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

ax = axes[0]
try:
    ax.plot(dates, spy_weights, label='SPY', lw=0.8)
    ax.plot(dates, ief_weights, label='IEF', lw=0.8)
    ax.set_xlabel('Date')
except Exception:
    ax.plot(spy_weights, label='SPY', lw=0.8)
    ax.plot(ief_weights, label='IEF', lw=0.8)
ax.axhline(0.5, color='k', lw=0.5, ls='--')
ax.set_title('Untrained Dirichlet Weights — N=2 (SPY + IEF)')
ax.set_ylabel('Weight')
ax.legend()

axes[1].plot(turnovers, lw=0.8, color='orange')
axes[1].set_title('Daily Turnover')
axes[1].set_ylabel('Turnover')

cumret = np.cumprod(1 + np.array(rewards)) - 1
axes[2].plot(cumret, lw=0.8, color='green')
axes[2].set_title('Cumulative Net Return (random policy)')
axes[2].set_ylabel('Cum. Return')

plt.tight_layout()
plt.savefig('../results/rl/n2_random_sanity.png', dpi=120)
plt.show()
print('Figure saved.')